In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.table(f"{catalog}.bronze.products")

#casting
df = (
    df.filter(col("product_id").isNotNull())
    .withColumn("product_name", trim(initcap(col("product_name"))))
    .withColumn("brand", trim(initcap(col("brand"))))
    .withColumn("category_l1", trim(initcap(col("category.l1"))))
    .withColumn("category_l2", trim(initcap(col("category.l2"))))
    .withColumn("pack_size", trim(col("pack.size")).cast("int"))
    .withColumn("pack_unit", trim(initcap(col("pack.unit"))))
    .withColumn("mrp", col("mrp").cast("decimal(10,2)"))
    .withColumn("selling_price", col("selling_price").cast("decimal(10,2)"))
    .withColumn("is_active", col("is_active").cast("boolean"))
    .withColumn("last_changed_snapshot_date", to_timestamp(col("snapshot_date")))
    )


df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.products")

In [0]:
from pyspark.sql.functions import *

source = spark.read.table(f"{catalog}.silver.products")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.product_scd1 (
    product_id STRING,
    product_name STRING,
    brand STRING,
    category_l1 STRING,
    category_l2 STRING,
    pack_size INT,
    pack_unit STRING,
    mrp DECIMAL(10,2),
    selling_price DECIMAL(10,2),
    is_active BOOLEAN,
    last_changed_snapshot_date TIMESTAMP
)
USING DELTA
""")

update_condition = """
    target.product_name <> source.product_name
 OR target.brand <> source.brand
 OR target.category_l1 <> source.category_l1
 OR target.category_l2 <> source.category_l2
 OR target.pack_size <> source.pack_size
 OR target.pack_unit <> source.pack_unit
 OR target.mrp <> source.mrp
 OR target.selling_price <> source.selling_price
 OR target.is_active <> source.is_active
 OR target.last_changed_snapshot_date <> source.last_changed_snapshot_date
"""

source.createOrReplaceTempView("source_products")

spark.sql(f"""
MERGE INTO {catalog}.silver.product_scd1 AS target
USING source_products AS source
ON target.product_id = source.product_id
WHEN MATCHED AND ({update_condition})
  THEN UPDATE SET
    product_name = source.product_name,
    brand = source.brand,
    category_l1 = source.category_l1,
    category_l2 = source.category_l2,
    pack_size  = source.pack_size,
    pack_unit = source.pack_unit,
    mrp  = source.mrp,
    selling_price  = source.selling_price,
    is_active = source.is_active,
    last_changed_snapshot_date = source.last_changed_snapshot_date
WHEN NOT MATCHED
  THEN INSERT *
""")